# Deep Learning Recommendation System

Build Neural Collaborative Filtering model using TensorFlow.

In [ ]:
import sys
sys.path.append('../src')
sys.path.append('../evaluation')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from metrics import rmse, mae
import matplotlib.pyplot as plt

## 1. Load and Prepare Data

In [ ]:
# Load data
ratings = pd.read_csv('../data/processed/ratings.csv')

print(f"Total ratings: {len(ratings)}")
print(f"Rating range: {ratings['rating'].min()} - {ratings['rating'].max()}")

## 2. Prepare Training Data

In [ ]:
# Encode user and item IDs
user_encoder = dict(zip(ratings['user_id'].unique(), range(ratings['user_id'].nunique())))
item_encoder = dict(zip(ratings['item_id'].unique(), range(ratings['item_id'].nunique())))

ratings['user_idx'] = ratings['user_id'].map(user_encoder)
ratings['item_idx'] = ratings['item_id'].map(item_encoder)

# Split data
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

# Prepare X and y
X_train = np.array([train_data['user_idx'].values, train_data['item_idx'].values]).T
y_train = train_data['rating'].values / 5.0  # Normalize to [0, 1]

X_test = np.array([test_data['user_idx'].values, test_data['item_idx'].values]).T
y_test = test_data['rating'].values / 5.0

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Unique users: {len(user_encoder)}")
print(f"Unique items: {len(item_encoder)}")

## 3. Build Neural Collaborative Filtering Model

In [ ]:
def build_ncf_model(n_users, n_items, embedding_dim=50):
    """Build Neural Collaborative Filtering model"""
    
    # User input and embedding
    user_input = layers.Input(shape=(1,), name='user_input')
    user_embedding = layers.Embedding(n_users, embedding_dim, name='user_embedding')(user_input)
    user_vec = layers.Flatten()(user_embedding)
    
    # Item input and embedding
    item_input = layers.Input(shape=(1,), name='item_input')
    item_embedding = layers.Embedding(n_items, embedding_dim, name='item_embedding')(item_input)
    item_vec = layers.Flatten()(item_embedding)
    
    # Concatenate embeddings
    concat = layers.Concatenate()([user_vec, item_vec])
    
    # Deep layers
    x = layers.Dense(256, activation='relu')(concat)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Dense(32, activation='relu')(x)
    
    # Output layer
    output = layers.Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=[user_input, item_input], outputs=output)
    
    return model

# Build model
ncf_model = build_ncf_model(len(user_encoder), len(item_encoder), embedding_dim=50)
ncf_model.summary()

## 4. Compile and Train Model

In [ ]:
# Compile model
ncf_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

# Callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Train model
history = ncf_model.fit(
    [X_train[:, 0], X_train[:, 1]], y_train,
    validation_data=([X_test[:, 0], X_test[:, 1]], y_test),
    epochs=50,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)

## 5. Evaluate Model

In [ ]:
# Make predictions
y_pred = ncf_model.predict([X_test[:, 0], X_test[:, 1]])
y_pred_unscaled = y_pred * 5.0  # Denormalize
y_test_unscaled = y_test * 5.0

# Calculate metrics
rmse_score = rmse(y_test_unscaled, y_pred_unscaled)
mae_score = mae(y_test_unscaled, y_pred_unscaled)

print(f"\nNeural Collaborative Filtering Results:")
print(f"RMSE: {rmse_score:.4f}")
print(f"MAE: {mae_score:.4f}")

## 6. Plot Training History

In [ ]:
# Plot training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Model Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Train MAE')
plt.plot(history.history['val_mae'], label='Val MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.title('Model MAE')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 7. Generate Recommendations

In [ ]:
def get_recommendations_ncf(model, user_idx, n_recommendations=10, 
                            reverse_item_encoder=None, rated_items=None):
    """Get top-N recommendations for a user"""
    
    n_items = len(reverse_item_encoder)
    
    # Predict ratings for all items
    user_indices = np.full(n_items, user_idx)
    item_indices = np.arange(n_items)
    
    predictions = model.predict([user_indices, item_indices], verbose=0).flatten()
    
    # Get top-N items (excluding already rated)
    if rated_items is not None:
        predictions[list(rated_items)] = -1  # Exclude rated items
    
    top_indices = np.argsort(predictions)[::-1][:n_recommendations]
    
    items = [reverse_item_encoder[idx] for idx in top_indices]
    scores = predictions[top_indices]
    
    return items, scores

# Create reverse encoder
reverse_item_encoder = {v: k for k, v in item_encoder.items()}
reverse_user_encoder = {v: k for k, v in user_encoder.items()}

# Get recommendations for sample users
for user_id in [1, 5, 10]:
    user_idx = user_encoder[user_id]
    
    # Get already rated items
    rated_items = set(ratings[ratings['user_id'] == user_id]['item_idx'].values)
    
    items, scores = get_recommendations_ncf(
        ncf_model, user_idx, n_recommendations=10,
        reverse_item_encoder=reverse_item_encoder,
        rated_items=rated_items
    )
    
    print(f"\nTop 10 recommendations for User {user_id}:")
    for i, (item, score) in enumerate(zip(items, scores), 1):
        print(f"{i}. Movie {item} (Score: {score*5:.2f}/5.0)")

## 8. Save Model

In [ ]:
# Save model
ncf_model.save('../data/models/ncf_model.h5')
print("Model saved successfully!")